In [1]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import os

# 1. Baca data TF-IDF hasil split sebelumnya
df_train = pd.read_csv("tfidf_training.csv")
df_test = pd.read_csv("tfidf_testing.csv")

# 2. Pisah fitur kata (X) dan label kelas (y)
X_train = df_train.drop(columns=['ID', 'Label'])
y_train = df_train['Label']  

X_test = df_test.drop(columns=['ID', 'Label'])
y_test = df_test['Label']    

print(f"Data Training: {X_train.shape[0]} baris x {X_train.shape[1]} kata unik")
print(f"Data Testing : {X_test.shape[0]} baris x {X_test.shape[1]} kata unik")
print(f"Distribusi Label Training: \n{y_train.value_counts()}")

Data Training: 160 baris x 7424 kata unik
Data Testing : 40 baris x 7424 kata unik
Distribusi Label Training: 
Label
1    80
2    80
Name: count, dtype: int64


In [2]:
# Evaluasi (7.424 kata unik) sebagai perbandingan 

# 1. Model kNN (k=5)
knn_raw = KNeighborsClassifier(n_neighbors=5)
knn_raw.fit(X_train, y_train)
acc_knn_raw = accuracy_score(y_test, knn_raw.predict(X_test))

# 2. Model Multinomial Naive Bayes 
mnb_raw = MultinomialNB()
mnb_raw.fit(X_train, y_train)
acc_mnb_raw = accuracy_score(y_test, mnb_raw.predict(X_test))

print(f"Akurasi kNN (Raw TF-IDF)        : {acc_knn_raw * 100:.2f}%")
print(f"Akurasi Naive Bayes (Raw TF-IDF): {acc_mnb_raw * 100:.2f}%")

Akurasi kNN (Raw TF-IDF)        : 100.00%
Akurasi Naive Bayes (Raw TF-IDF): 100.00%


In [3]:
# Daftar variasi reduksi dimensi PCA yang akan dilakukan
list_komponen = [150, 100, 50, 20, 10]
hasil_pca = []

# Masukkan baseline ke dalam tabel rekap
hasil_pca.append({
    "Metode": "Raw TF-IDF (Tanpa Reduksi)",
    "Jumlah Komponen": 7424,
    "Informasi Variansi (%)": 100.0,
    "Akurasi kNN (%)": round(acc_knn_raw * 100, 2),
    "Akurasi Naive Bayes (%)": round(acc_mnb_raw * 100, 2)
})

for n in list_komponen:
    print(f"--- Memproses & Menyimpan PCA {n} Komponen ---")
    
    # 1. Penerapan PCA
    pca = PCA(n_components=n, random_state=42)
    X_tr_pca = pca.fit_transform(X_train)  # Training di-fit_transform
    X_te_pca = pca.transform(X_test)       # Testing di-transform
    
    # 2. Hitung total variansi
    var_total = sum(pca.explained_variance_ratio_) * 100
    
    # 3. Buat DataFrame lengkap: ID | PC1 | PC2 | ... | Label 
    nama_kolom = [f"PC{i+1}" for i in range(n)]
    
    # DataFrame Training
    df_pca_train = pd.DataFrame(X_tr_pca, columns=nama_kolom)
    df_pca_train.insert(0, 'ID', df_train['ID'].values)
    df_pca_train['Label'] = df_train['Label'].values 
    
    # DataFrame Testing
    df_pca_test = pd.DataFrame(X_te_pca, columns=nama_kolom)
    df_pca_test.insert(0, 'ID', df_test['ID'].values)
    df_pca_test['Label'] = df_test['Label'].values    
    
    # 4. SIMPAN KE FILE .CSV DAN .XLSX
    file_tr_csv = f"pca_{n}_training.csv"
    file_tr_xlsx = f"pca_{n}_training.xlsx"
    df_pca_train.to_csv(file_tr_csv, index=False)
    df_pca_train.to_excel(file_tr_xlsx, index=False)
    
    file_te_csv = f"pca_{n}_testing.csv"
    file_te_xlsx = f"pca_{n}_testing.xlsx"
    df_pca_test.to_csv(file_te_csv, index=False)
    df_pca_test.to_excel(file_te_xlsx, index=False)
    
    print(f"  -> Berhasil simpan: {file_tr_csv} & {file_te_csv}")
    
    # 5. Evaluasi Akurasi kNN & Naive Bayes
    knn = KNeighborsClassifier(n_neighbors=5).fit(X_tr_pca, y_train)
    acc_knn = accuracy_score(y_test, knn.predict(X_te_pca)) * 100
    
    gnb = GaussianNB().fit(X_tr_pca, y_train)
    acc_gnb = accuracy_score(y_test, gnb.predict(X_te_pca)) * 100
    
    hasil_pca.append({
        "Metode": f"PCA {n} Komponen",
        "Jumlah Komponen": n,
        "Informasi Variansi (%)": round(var_total, 2),
        "Akurasi kNN (%)": round(acc_knn, 2),
        "Akurasi Naive Bayes (%)": round(acc_gnb, 2)
    })

# Tampilkan tabel rekap 
df_hasil_pca = pd.DataFrame(hasil_pca)
display(df_hasil_pca)

--- Memproses & Menyimpan PCA 150 Komponen ---
  -> Berhasil simpan: pca_150_training.csv & pca_150_testing.csv
--- Memproses & Menyimpan PCA 100 Komponen ---
  -> Berhasil simpan: pca_100_training.csv & pca_100_testing.csv
--- Memproses & Menyimpan PCA 50 Komponen ---
  -> Berhasil simpan: pca_50_training.csv & pca_50_testing.csv
--- Memproses & Menyimpan PCA 20 Komponen ---
  -> Berhasil simpan: pca_20_training.csv & pca_20_testing.csv
--- Memproses & Menyimpan PCA 10 Komponen ---
  -> Berhasil simpan: pca_10_training.csv & pca_10_testing.csv


,Metode,Jumlah Komponen,Informasi Variansi (%),Akurasi kNN (%),Akurasi Naive Bayes (%)
0,Raw TF-IDF (Tanpa Reduksi),7424,100.00,100.0,100.0
1,PCA 150 Komponen,150,98.73,95.0,62.5
2,PCA 100 Komponen,100,80.48,87.5,77.5
3,PCA 50 Komponen,50,52.34,100.0,72.5
4,PCA 20 Komponen,20,29.42,95.0,72.5
5,PCA 10 Komponen,10,18.66,97.5,90.0


In [4]:
# Seleksi Fitur Untuk Pemanfaatan Reduksi Dimensi Ribuan (Bertahap)

list_k_kata = [6000, 4000, 2000, 1000, 500]
hasil_seleksi = []

hasil_seleksi.append({
    "Tahap": "Data Asli (7.424 kata)",
    "Jumlah Kata": 7424,
    "Akurasi kNN (%)": round(acc_knn_raw * 100, 2),
    "Akurasi Naive Bayes (%)": round(acc_mnb_raw * 100, 2)
})

for k in list_k_kata:
    print(f"--- Memproses Seleksi Fitur Top-{k} Kata ---")
    skb = SelectKBest(chi2, k=k)
    X_tr_k = skb.fit_transform(X_train, y_train)
    X_te_k = skb.transform(X_test)
    
    # Ambil nama kata yang terpilih
    kata_terpilih = X_train.columns[skb.get_support()]
    
    # Buat DataFrame dengan kata terpilih + ID + Label
    df_chi_train = pd.DataFrame(X_tr_k, columns=kata_terpilih)
    df_chi_train.insert(0, 'ID', df_train['ID'].values)
    df_chi_train['Label'] = df_train['Label'].values
    
    df_chi_test = pd.DataFrame(X_te_k, columns=kata_terpilih)
    df_chi_test.insert(0, 'ID', df_test['ID'].values)
    df_chi_test['Label'] = df_test['Label'].values
    
    # SIMPAN KE .CSV DAN .XLSX
    file_chi_tr_csv = f"chi2_{k}_training.csv"
    file_chi_tr_xlsx = f"chi2_{k}_training.xlsx"
    df_chi_train.to_csv(file_chi_tr_csv, index=False)
    df_chi_train.to_excel(file_chi_tr_xlsx, index=False)
    
    file_chi_te_csv = f"chi2_{k}_testing.csv"
    file_chi_te_xlsx = f"chi2_{k}_testing.xlsx"
    df_chi_test.to_csv(file_chi_te_csv, index=False)
    df_chi_test.to_excel(file_chi_te_xlsx, index=False)
    
    print(f"  -> Berhasil simpan: {file_chi_tr_csv} & {file_chi_te_csv}")
    
    # Uji Akurasi
    knn = KNeighborsClassifier(n_neighbors=5).fit(X_tr_k, y_train)
    acc_knn = accuracy_score(y_test, knn.predict(X_te_k)) * 100
    
    mnb = MultinomialNB().fit(X_tr_k, y_train)
    acc_mnb = accuracy_score(y_test, mnb.predict(X_te_k)) * 100
    
    hasil_seleksi.append({
        "Tahap": f"Chi-Square Top-{k}",
        "Jumlah Kata": k,
        "Akurasi kNN (%)": round(acc_knn, 2),
        "Akurasi Naive Bayes (%)": round(acc_mnb, 2)
    })

df_hasil_seleksi = pd.DataFrame(hasil_seleksi)
display(df_hasil_seleksi)

--- Memproses Seleksi Fitur Top-6000 Kata ---
  -> Berhasil simpan: chi2_6000_training.csv & chi2_6000_testing.csv
--- Memproses Seleksi Fitur Top-4000 Kata ---
  -> Berhasil simpan: chi2_4000_training.csv & chi2_4000_testing.csv
--- Memproses Seleksi Fitur Top-2000 Kata ---
  -> Berhasil simpan: chi2_2000_training.csv & chi2_2000_testing.csv
--- Memproses Seleksi Fitur Top-1000 Kata ---
  -> Berhasil simpan: chi2_1000_training.csv & chi2_1000_testing.csv
--- Memproses Seleksi Fitur Top-500 Kata ---
  -> Berhasil simpan: chi2_500_training.csv & chi2_500_testing.csv


,Tahap,Jumlah Kata,Akurasi kNN (%),Akurasi Naive Bayes (%)
0,Data Asli (7.424 kata),7424,100.0,100.0
1,Chi-Square Top-6000,6000,100.0,100.0
2,Chi-Square Top-4000,4000,100.0,100.0
3,Chi-Square Top-2000,2000,100.0,100.0
4,Chi-Square Top-1000,1000,77.5,100.0
5,Chi-Square Top-500,500,90.0,100.0
